# TML Assignment 3 - Robustness
## resnet34 + TRADES(beta=3) + EMA + CutMix + long SGDR (120 epochs)

**The bet: expand the robust frontier with CutMix + weight averaging + long
training (Rebuffi et al. 2021, "Fixing Data Augmentation to Improve Adversarial
Robustness").** This is the one documented way to get MORE robust accuracy from the
SAME data (our hard constraint - no extra/synthetic data allowed).

**Why this is different from the RandomErasing that failed (Exp19, robust collapsed
0.447->0.361):** RandomErasing *zeroes* a patch, destroying information - the PGD
attack and the TRADES consistency target were both built on a corrupted image, and
eval images are never erased, so robustness collapsed. **CutMix instead recomposes
two REAL images** (pastes a box from image B into image A) and mixes their labels by
area. No information is destroyed; the composite is locally in-distribution. The
adversarial signal stays intact.

**Why it needs the compute you now have:** Rebuffi shows CutMix only helps AT *with*
(a) model weight averaging - we have EMA(0.999) - and (b) LONG training, because the
harder augmented objective converges later. Hence 120 epochs with SGDR restarts
[40,60,80,100,120] (the proven 0.595 schedule + two more 20-epoch restarts).

**Implementation (correct TRADES+CutMix coupling):**
- natural term: lam*CE(f(x_mix), y_a) + (1-lam)*CE(f(x_mix), y_b)
- inner PGD attack maximizes the same mixed CE (so x_adv targets the mixed labels)
- robust term: KL(f(x_adv) || f(x_mix)) - unchanged, label-free, self-consistent
- CUTMIX_PROB=0.5 (half the batches get CutMix), alpha=1.0 (lam ~ Uniform)

Baseline to beat: resnet34 + SGDR-3 (no CutMix) = clean 0.732 / PGD-20 0.457 /
**0.595 leaderboard**. Everything else identical to that run (beta=3, EMA 0.999,
lr0.1 warmup3, CE-PGD7 eps8/255, crop+flip, best-ckpt by val_score, 48k/2k split).

NOTE: ~120 epochs x ~60s = ~2h on a T4 - run on the cluster, or on Colab with
RESUME=True (re-run the cell to resume if disconnected). Run cells top to bottom.


## 1. Setup

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/tml_assignment3'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)

## 2. Download dataset

In [ ]:
DATA_PATH = '/content/train.npz'
if not os.path.exists(DATA_PATH):
    !wget -q -O {DATA_PATH} "https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz"
print('Downloaded:', os.path.exists(DATA_PATH), os.path.getsize(DATA_PATH) / 1e6, 'MB')

## 3. Imports & hyperparameters

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.models import resnet34
import torchvision.transforms.v2 as T
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True  # speeds up conv kernels for fixed input size

# ---- Hyperparameters (v2's recipe + TRADES) ----
NUM_CLASSES = 9
MODEL_NAME = 'resnet34'
BATCH_SIZE = 128
EPOCHS = 120
WARMUP_EPOCHS = 3
VAL_SIZE = 2000
SEED = 42

# Optimizer + LR schedule: warmup -> cosine, peak lr=0.1 (v2's recipe - confirmed
# better than lr=0.02).
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# PGD attack (training) - identical to all previous experiments (Madry et al., 2018).
# Adversarial examples are generated with this CE-based attack (NOT a KL-based
# inner max), avoiding fp16 KL computation inside the PGD loop.
EPS = 8 / 255
ALPHA = 2 / 255
PGD_STEPS_TRAIN = 7

# PGD attack (evaluation - stronger)
PGD_STEPS_EVAL = 20

# TRADES: outer loss = CE(f(x), y) + TRADES_BETA * KL(f(x_adv) || f(x)), KL in fp32.
# beta=3 (vs. the failed beta=6) biases the tradeoff toward clean accuracy.
TRADES_BETA = 3.0

# CutMix (Rebuffi et al. 2021): prob of applying CutMix per batch, and the
# Beta(alpha,alpha) sampling for lam. alpha=1 -> lam ~ Uniform(0,1).
CUTMIX_PROB = 0.5
CUTMIX_ALPHA = 1.0

# EMA (exponential moving average) of model weights - applied to all floating-point
# params/buffers (incl. BatchNorm running stats) after every optimizer step.
# Validation/checkpointing/submission all use ema_model.
EMA_DECAY = 0.999

# 3-cycle SGDR: epochs [0,40) -> 40-epoch cosine, epochs [40,60) -> 20-epoch
# cosine restart (reproduces Exp9/10's ~0.5859 result), epochs [60,80) ->
# second 20-epoch cosine restart.
LR_CYCLE_BOUNDARIES = [40, 60, 80, 100, 120]

# Mixed precision (big speedup on T4 tensor cores; PGD steps dominate the per-batch cost)
USE_AMP = (device.type == 'cuda')

RESUME = True
CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r34_cutmix_sgdr.pt')
BEST_CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_r34_cutmix_sgdr_best.pt')

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Device:', device, '| AMP:', USE_AMP)

## 4. Data loading

In [ ]:
data = np.load(DATA_PATH)
images = torch.from_numpy(data['images']).float() / 255.0  # (N, 3, 32, 32) in [0,1]
labels = torch.from_numpy(data['labels']).long()
print('Images:', images.shape, 'Labels:', labels.shape)

full_dataset = TensorDataset(images, labels)
train_size = len(full_dataset) - VAL_SIZE
train_set, val_set = random_split(
    full_dataset, [train_size, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)
print('Train size:', len(train_set), 'Val size:', len(val_set))

# Standard CIFAR-style augmentation (applied to the clean image before PGD attack)
augment = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

# num_workers=0: data is already an in-memory TensorDataset
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

## 5. Model

In [ ]:
def build_model():
    model = resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_model().to(device)

ema_model = build_model().to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)
ema_model.eval()

# sanity check matching task_template.py
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES), out.shape
print('Output shape OK:', out.shape)

## 6. PGD attack (identical to all previous PGD-AT experiments)

In [ ]:
def pgd_attack(model, x, y, eps, alpha, steps, random_start=True, y_b=None, lam=1.0):
    """Untargeted L_inf PGD maximizing (CutMix-mixed) cross-entropy. x in [0,1].
    When y_b is None this is the standard single-label attack used by eval."""
    was_training = model.training
    model.eval()

    x_orig = x.detach()
    if random_start:
        delta = torch.empty_like(x_orig).uniform_(-eps, eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()
    else:
        x_adv = x_orig.clone()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x_adv)
            if y_b is None:
                loss = F.cross_entropy(logits, y)
            else:
                loss = lam * F.cross_entropy(logits, y) + (1.0 - lam) * F.cross_entropy(logits, y_b)
        grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(x_adv, x_orig - eps, x_orig + eps)
            x_adv = torch.clamp(x_adv, 0.0, 1.0)

    if was_training:
        model.train()
    return x_adv.detach()


def fgsm_attack(model, x, y, eps):
    was_training = model.training
    model.eval()
    x_orig = x.detach().requires_grad_(True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        logits = model(x_orig)
        loss = F.cross_entropy(logits, y)
    grad = torch.autograd.grad(loss, x_orig)[0]
    x_adv = torch.clamp(x_orig + eps * grad.sign(), 0.0, 1.0).detach()
    if was_training:
        model.train()
    return x_adv


def rand_bbox(H, W, lam):
    """Random box whose area ~= (1-lam) of the image (standard CutMix)."""
    cut_rat = math.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = int(np.clip(cx - cut_w // 2, 0, W))
    bby1 = int(np.clip(cy - cut_h // 2, 0, H))
    bbx2 = int(np.clip(cx + cut_w // 2, 0, W))
    bby2 = int(np.clip(cy + cut_h // 2, 0, H))
    return bbx1, bby1, bbx2, bby2


def cutmix_batch(x, y, alpha, prob):
    """With prob `prob`, paste a random box from a shuffled copy of the batch into x,
    returning (x_mixed, y_a, y_b, lam) with lam = area-corrected weight of the base
    image. Otherwise a no-op (y_b=y_a, lam=1). Recomposes two REAL images (no info
    destroyed) - unlike RandomErasing - so it does not corrupt the adversarial signal."""
    if np.random.rand() > prob:
        return x, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3)
    bbx1, bby1, bbx2, bby2 = rand_bbox(H, W, lam)
    x[:, :, bby1:bby2, bbx1:bbx2] = x[perm, :, bby1:bby2, bbx1:bbx2]
    lam = 1.0 - ((bbx2 - bbx1) * (bby2 - bby1) / (H * W))
    return x, y, y[perm], lam

## 7. Optimizer, LR schedule (warmup + cosine), training loop

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    # SGDR-style cosine restarts: for epoch < LR_CYCLE_BOUNDARIES[i], the cosine
    # formula uses "total epochs" = LR_CYCLE_BOUNDARIES[i]. With
    # LR_CYCLE_BOUNDARIES = [40, 60, 80], lr does a 40-epoch cosine, then jumps
    # back up and does a 20-epoch cosine (epochs 40-59), then jumps back up
    # again and does another 20-epoch cosine (epochs 60-79).
    total = next((b for b in LR_CYCLE_BOUNDARIES if epoch < b), LR_CYCLE_BOUNDARIES[-1])
    progress = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
best_score = -1.0
if RESUME and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    ema_model.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Starting fresh training')

if RESUME and os.path.exists(BEST_CKPT_PATH):
    best_score = torch.load(BEST_CKPT_PATH, map_location='cpu').get('score', -1.0)
    print(f'Loaded best score so far: {best_score:.4f}')

In [ ]:
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total


def evaluate_pgd(model, loader, eps, alpha, steps, max_batches=None):
    model.eval()
    correct, total = 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, eps, alpha, steps)
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x_adv).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
@torch.no_grad()
def update_ema(ema_model, model, decay):
    msd = model.state_dict()
    for k, v in ema_model.state_dict().items():
        if v.dtype.is_floating_point:
            v.mul_(decay).add_(msd[k], alpha=1 - decay)
        else:
            v.copy_(msd[k])


for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    running_loss, running_correct, running_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        x = augment(x)

        # CutMix (Rebuffi et al. 2021): recompose two real images, mix labels by area.
        x, y_a, y_b, lam = cutmix_batch(x, y, CUTMIX_ALPHA, CUTMIX_PROB)

        # Adversarial examples via CE-based PGD, targeting the CutMix-mixed labels.
        x_adv = pgd_attack(model, x, y_a, EPS, ALPHA, PGD_STEPS_TRAIN, y_b=y_b, lam=lam)

        model.train()
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits_clean = model(x)
            logits_adv = model(x_adv)
            loss_natural = lam * F.cross_entropy(logits_clean, y_a) + (1.0 - lam) * F.cross_entropy(logits_clean, y_b)

        # TRADES KL term in fp32, logits clamped to [-30,30] to avoid 0*-inf=nan.
        logits_adv_f = logits_adv.float().clamp(-30, 30)
        logits_clean_f = logits_clean.float().clamp(-30, 30)
        log_p_adv = F.log_softmax(logits_adv_f, dim=1)
        p_clean = F.softmax(logits_clean_f, dim=1)
        loss_robust = F.kl_div(log_p_adv, p_clean, reduction='batchmean')

        loss = loss_natural + TRADES_BETA * loss_robust

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        update_ema(ema_model, model, EMA_DECAY)

        running_loss += loss.item() * y.size(0)
        running_correct += (logits_adv.argmax(dim=1) == y_a).sum().item()
        running_total += y.size(0)

        pbar.set_postfix(loss=running_loss / running_total, adv_acc=running_correct / running_total)

    scheduler.step()

    train_loss = running_loss / running_total
    train_adv_acc = running_correct / running_total
    val_clean_acc = evaluate_clean(ema_model, val_loader)
    val_pgd_acc = evaluate_pgd(ema_model, val_loader, EPS, ALPHA, PGD_STEPS_TRAIN, max_batches=5)
    val_score = 0.5 * val_clean_acc + 0.5 * val_pgd_acc

    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | lr {scheduler.get_last_lr()[0]:.4f} | '
          f'train_loss {train_loss:.4f} | train_adv_acc {train_adv_acc:.4f} | '
          f'val_clean_acc(ema) {val_clean_acc:.4f} | val_pgd7_acc(ema) {val_pgd_acc:.4f} | '
          f'val_score(ema) {val_score:.4f} | {dt:.1f}s')

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'val_clean_acc': val_clean_acc,
        'val_pgd7_acc': val_pgd_acc,
        'score': val_score,
    }
    torch.save(ckpt, CKPT_PATH)

    if val_score > best_score:
        best_score = val_score
        torch.save(ckpt, BEST_CKPT_PATH)
        print(f'  -> new best (val_score={val_score:.4f}), saved to {BEST_CKPT_PATH}')

## 8. Final evaluation (best checkpoint, stronger PGD-20)

In [ ]:
# Load the best checkpoint by val_score (not necessarily the last epoch)
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_ckpt['ema_state_dict'])
print(f"Loaded best checkpoint (EMA weights) from epoch {best_ckpt['epoch']+1} (val_score={best_ckpt['score']:.4f})")

final_clean_acc = evaluate_clean(model, val_loader)
final_pgd20_acc = evaluate_pgd(model, val_loader, EPS, ALPHA, PGD_STEPS_EVAL)

model.eval()
fgsm_correct, fgsm_total = 0, 0
for x, y in val_loader:
    x, y = x.to(device), y.to(device)
    x_adv = fgsm_attack(model, x, y, EPS)
    with torch.no_grad():
        pred = model(x_adv).argmax(dim=1)
    fgsm_correct += (pred == y).sum().item()
    fgsm_total += y.size(0)
final_fgsm_acc = fgsm_correct / fgsm_total

print(f'Final clean accuracy:  {final_clean_acc:.4f}')
print(f'Final FGSM accuracy:   {final_fgsm_acc:.4f} (eps={EPS:.4f})')
print(f'Final PGD-20 accuracy: {final_pgd20_acc:.4f} (eps={EPS:.4f}, alpha={ALPHA:.4f})')
print(f'Estimated score (0.5*clean + 0.5*pgd20): {0.5*final_clean_acc + 0.5*final_pgd20_acc:.4f}')

## 9. Save submission state dict

In [ ]:
SUBMIT_PATH = os.path.join(PROJECT_DIR, f'{MODEL_NAME}_r34_cutmix_sgdr_submission.pt')

# sanity checks matching task_template.py / submission.py requirements
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES)

torch.save(model.state_dict(), SUBMIT_PATH)
print('Saved submission state dict to:', SUBMIT_PATH)
print('Model name for submission.py:', MODEL_NAME)